# Audio Fundamentals: From Waveforms to Transformers

**Goal:** Understand what each tensor means as audio travels from a sampled waveform to a small, untrained audio classifier. We are *not* training a speech recognizer here.

**What you will learn:** sampling → waveform → time–frequency analysis (STFT) → Mel spectrogram → audio-frame embeddings → positions → self-attention → classification logits.

**How to use this notebook:** Run cells from top to bottom. Read each short explanation *before* its code cell. Change one parameter at a time and compare plots/shapes. Everything runs on CPU; an A100 is unnecessary.

> **Important:** The last projection, positional embedding, attention, and classifier layers have **random weights**. Their output shapes are meaningful, but the output values are **not meaningful predictions** until training.

## 0. Imports and reproducibility

- `torch` creates and transforms numeric tensors.
- `torchaudio` provides standard audio transformations, including Mel filterbanks.
- `matplotlib` plots signals.
- We set a random seed so randomly initialized layers give repeatable results.

If `torchaudio` is unavailable, install a version compatible with your PyTorch build. In Colab, it is typically preinstalled.

In [ ]:
import torch
import torch.nn as nn
import torchaudio
import matplotlib.pyplot as plt

torch.manual_seed(42)
print('PyTorch:', torch.__version__)
print('torchaudio:', torchaudio.__version__)

## 1. Sampling and waveforms

A **waveform** is a sequence of signal-amplitude measurements taken at regular time intervals.

- **Sampling rate**: number of measurements per second. `16_000` means 16,000 measurements every second.
- **Duration**: recording length in seconds.
- **Frequency**: for a pure sine tone, how many oscillations happen every second. 440 Hz means 440 cycles per second.
- **Amplitude**: height of the wave; it is not the same thing as frequency.

For 1 second at 16 kHz, the resulting 1-D tensor has `16_000` values. The formula below makes a *synthetic tone*, not recorded speech:

\[x(t)=\sin(2\pi f t)\]

We plot the **first 400 samples only** (~25 ms), because plotting the full second makes individual oscillations hard to see.

In [ ]:
sample_rate = 16_000
duration = 1.0
frequency = 440

# Times: 0, 1/16000, 2/16000, ... seconds
t = torch.arange(int(sample_rate * duration)) / sample_rate

# One second of a 440 Hz tone
waveform = torch.sin(2 * torch.pi * frequency * t)

print('Time tensor:', t.shape)
print('Waveform tensor:', waveform.shape)
print('First five samples:', waveform[:5])

plt.figure(figsize=(10, 3))
plt.plot(t[:400], waveform[:400])
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.title('440 Hz sine wave — first 400 samples (~25 ms)')
plt.grid(alpha=0.2)
plt.show()

### Check your understanding

- If we changed `duration` to `2`, the tensor would have **32,000 samples**.
- If we changed `frequency` to `880` while keeping duration and sample rate constant, the tensor would **still have 16,000 samples**, but it would oscillate twice as fast.

**Try:** Change `frequency` above from 440 to 880 and compare the waveform over the same first 400 samples.

## 2. Two different meanings of “two tones”

These are easy to mix up:

**Mixing (addition):** `tone_440 + 0.5 * tone_880` means both frequencies play **at the same time**. The recording lasts **one second**.

**Concatenation:** `torch.cat((tone_440, 0.5 * tone_880))` means 440 Hz plays **first**, then 880 Hz plays **afterward**. The recording lasts **two seconds**.

The `0.5` reduces the second tone's amplitude. It does *not* change its frequency. We'll use **concatenation** below because it makes the STFT's time information easy to see.

In [ ]:
sr = 16_000
t = torch.arange(sr) / sr  # one second

tone_440 = torch.sin(2 * torch.pi * 440 * t)
tone_880 = 0.5 * torch.sin(2 * torch.pi * 880 * t)

mixed = tone_440 + tone_880        # 1 second: both simultaneously
concatenated = torch.cat((tone_440, tone_880))  # 2 seconds: one after the other

print('Mixed:', mixed.shape, 'duration:', mixed.numel() / sr, 'seconds')
print('Concatenated:', concatenated.shape, 'duration:', concatenated.numel() / sr, 'seconds')

# Use this as the audio for the rest of the notebook.
waveform = concatenated

time_axis = torch.arange(waveform.numel()) / sr
plt.figure(figsize=(10, 3))
plt.plot(time_axis, waveform, linewidth=0.6)
plt.axvline(1.0, color='black', linestyle='--', alpha=0.5, label='tone changes')
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.title('Two-second waveform: 440 Hz then 880 Hz')
plt.legend()
plt.show()

## 3. STFT: frequency analysis in small windows

A waveform tells us amplitude **over time**. A Fourier transform tells us **which frequencies** a chunk contains. A **Short-Time Fourier Transform (STFT)** repeatedly applies that analysis to short, overlapping chunks so we can see **which frequencies occurred when**.

Here are the arguments:

| Argument | What it means in this notebook |
|---|---|
| `waveform` | Your amplitude samples (2 seconds, 32,000 samples). |
| `n_fft=400` | FFT size. With a 400-sample window, each analysis spans **25 ms** at 16 kHz. |
| `hop_length=160` | Move forward **160 samples**, or **10 ms**, between successive analyses. |
| `window=torch.hann_window(400)` | Multiply each chunk by a smooth Hann taper, reducing artificial boundary effects called *spectral leakage*. |
| `return_complex=True` | Preserve a complex number for each frequency bin: magnitude and phase information. |

**Overlap:** a 400-sample window moved by 160 samples overlaps its neighbor by **240 samples**.

**Why 201 frequency bins?** For 400-point FFTs on real-valued audio, we keep the nonnegative-frequency bins: `400 // 2 + 1 = 201`.

**Why 201 time frames here?** This recording has 32,000 samples (2 seconds). PyTorch defaults to `center=True`, which pads the signal, giving `1 + 32000 // 160 = 201` time frames. A one-second waveform would instead produce 101 frames.

In [ ]:
stft = torch.stft(
    waveform,
    n_fft=400,
    hop_length=160,
    window=torch.hann_window(400),
    return_complex=True,
)

# Absolute value discards phase and keeps magnitude.
spectrogram = stft.abs()

print('STFT complex tensor:', stft.shape, stft.dtype)
print('Magnitude spectrogram:', spectrogram.shape)
print('Frequency bins:', spectrogram.shape[0])
print('Time frames:', spectrogram.shape[1])

# Compute axes from the actual tensor and sample rate: no hardcoded 1-second extent.
freqs = torch.fft.rfftfreq(400, d=1 / sr)
frame_times = torch.arange(spectrogram.shape[1]) * 160 / sr

plt.figure(figsize=(10, 4))
plt.pcolormesh(
    frame_times.numpy(),
    freqs.numpy(),
    torch.log1p(spectrogram).numpy(),
    shading='auto',
)
plt.ylim(0, 1500)
plt.xlabel('Time (seconds)')
plt.ylabel('Frequency (Hz)')
plt.title('STFT spectrogram: 440 Hz first, 880 Hz second')
plt.colorbar(label='log(1 + magnitude)')
plt.show()

### How to read the spectrogram

- **X-axis = time**: left to right through the two-second recording.
- **Y-axis = frequency**: low frequencies at the bottom, high at the top.
- **Brightness = strength** of a frequency within that time window.
- Expect a strong horizontal band near **440 Hz for the first second**, then near **880 Hz for the second second**.

The transition near one second may look slightly blurred because neighboring STFT windows overlap. Spectrogram values here are **magnitudes**, not probabilities.

**Try:** Change `waveform = concatenated` to `waveform = mixed` in section 2 and rerun. You should see **both frequency bands present throughout one second**, not one after the other.

## 4. Mel spectrogram: a different frequency scale

The STFT uses equally spaced **frequency bins** (in hertz). A **Mel filterbank** combines those frequency bins into bands designed around aspects of human pitch perception: more detail at lower frequencies, less at higher frequencies.

Here we ask for **80 Mel bands**. This does *not* mean 80 audio samples or 80 words. Each Mel band measures the energy in a particular range of frequencies.

`torchaudio.transforms.MelSpectrogram` does the STFT and Mel filterbank internally. Its default output is **power** (squared magnitude) on the Mel scale. A log transform is often used for better numerical/dynamic-range handling and visualization.

With our **two-second** waveform and STFT settings, expect **[80, 201]**: 80 Mel bands across 201 time frames.

In [ ]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=sr,
    n_fft=400,
    hop_length=160,
    n_mels=80,
)

mel = mel_transform(waveform)
log_mel = torch.log10(mel + 1e-9)

print('Waveform:', waveform.shape)
print('Mel power spectrogram:', mel.shape)
print('Log-Mel spectrogram:', log_mel.shape)

plt.figure(figsize=(10, 4))
plt.imshow(
    log_mel.numpy(),
    aspect='auto',
    origin='lower',
    extent=[0, waveform.numel() / sr, 0, 80],
)
plt.xlabel('Time (seconds)')
plt.ylabel('Mel band index (not Hz)')
plt.title('Log-Mel spectrogram of the same two-second waveform')
plt.colorbar(label='log10(Mel power + 1e-9)')
plt.show()

### STFT versus Mel — what changed?

- STFT magnitude: `[201, 201]` = **201 linear-frequency bins × 201 time frames**.
- Mel spectrogram: `[80, 201]` = **80 perceptually spaced bands × 201 time frames**.
- The **time dimension stays the same**, because `n_fft` and `hop_length` were unchanged.
- Mel features are continuous values. **They are not discrete token IDs.**

**Check:** What would change if you set `n_mels=40`? The first dimension would become 40; the number of frames would remain 201.

## 5. Transposing: make a sequence of audio feature vectors

So far, `mel` has shape `[80, 201]` = `[Mel bands, time frames]`.

For a Transformer, it is intuitive to treat **each time frame as one sequence element**, with **80 values describing its frequency content**.

We transpose to `[201, 80]`: **201 frames, each containing 80 numbers**. A frame is analogous to a token *position*, but it is **not** a word or discrete text token.

In [ ]:
audio_frames = log_mel.transpose(0, 1)
print('Before transpose [mel bands, time]:', log_mel.shape)
print('After transpose [time, mel bands]:', audio_frames.shape)
print('One frame feature vector:', audio_frames[0].shape)
print('First five values of frame 0:', audio_frames[0, :5])

## 6. Project each audio frame into the Transformer hidden dimension

Our frames have **80 numbers**, but let's build a Transformer whose hidden dimension is **256**.

`nn.Linear(80, 256)` learns a matrix that maps each 80-feature frame into a 256-feature **embedding**. It applies the **same mapping to every time frame**.

- Before: `[201, 80]`
- After: `[201, 256]`

**Important:** This is not the same as a text model's integer token-ID embedding lookup. These are **continuous input features** being linearly projected.

Since the layer has not been trained, its current weights are random. We are learning tensor plumbing, not semantic audio recognition yet.

In [ ]:
projection = nn.Linear(80, 256)
embeddings = projection(audio_frames)

print('Input frames:', audio_frames.shape)
print('Projected embeddings:', embeddings.shape)
print('Projection weight shape:', projection.weight.shape)  # [256, 80]

## 7. Add positional information

Self-attention alone does not know the order of the sequence. But in audio, **which sound came before another** matters.

`nn.Embedding(number_of_frames, 256)` creates one learned vector for every possible frame position. We add that to its corresponding acoustic embedding.

- Acoustic vector: *what this time frame sounds like*.
- Position vector: *where this frame is in the sequence*.
- Shape stays `[201, 256]`.

We derive the number of positions from `embeddings.shape[0]` rather than hardcoding 101, because this recording is two seconds long.

In [ ]:
num_frames = embeddings.shape[0]  # 201 for two seconds
position_embedding = nn.Embedding(num_frames, 256)
positions = torch.arange(num_frames)

embeddings_with_position = embeddings + position_embedding(positions)
print('Audio + positional embeddings:', embeddings_with_position.shape)

## 8. Self-attention: audio frames share context

Audio sounds overlap across time. The meaning of one short frame can depend on nearby and distant frames. **Self-attention** lets a frame gather information from other frames.

`nn.MultiheadAttention(embed_dim=256, num_heads=8)` applies eight attention heads to 256-dimensional frame embeddings.

The module expects a **batch dimension** because we set `batch_first=True`:

`[time=201, features=256]` → `[batch=1, time=201, features=256]`.

Calling `attention(x, x, x)` means the **queries, keys, and values** all originate from the audio frame sequence. The output retains the same shape, but each frame's vector can incorporate information from other frames.

This is one attention layer only, *not* a complete trained Transformer encoder with multiple blocks.

In [ ]:
attention = nn.MultiheadAttention(
    embed_dim=256,
    num_heads=8,
    batch_first=True,
)

x = embeddings_with_position.unsqueeze(0)  # add batch dimension
output, weights = attention(x, x, x)

print('Attention input:', x.shape)
print('Attention output:', output.shape)
print('Attention weights:', weights.shape)  # [batch, query time, key time]

### Optional: inspect the attention weights

`weights[0]` is a **201 × 201 matrix**. Entry `(i, j)` indicates how strongly the representation at frame `i` attends to frame `j`, averaged across heads by default.

The attention patterns below come from **random, untrained weights**. They are a structural illustration, **not interpretable evidence** that the model understands the audio.

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(weights[0].detach().numpy(), aspect='auto', origin='lower', cmap='viridis')
plt.xlabel('Key frame index')
plt.ylabel('Query frame index')
plt.title('Attention weights (randomly initialized)')
plt.colorbar(label='Attention weight')
plt.show()

## 9. Classification head: turn the sequence into logits

A classification task asks one question about the **entire recording**, such as: *Is this speech, music, or noise?*

The attention layer gives one vector per frame: `[1, 201, 256]`. We reduce the time dimension by averaging all 201 frame vectors:

`[1, 201, 256]` → `[1, 256]`.

Then a linear classifier with 3 outputs gives a score (a **logit**) for each class:

`[1, 256]` → `[1, 3]`.

**A logit is not a probability.** `softmax` can convert logits to values between 0 and 1 that sum to 1. But because this model was **not trained**, neither the logits nor the probabilities should be treated as valid classification predictions.

In [ ]:
pooled = output.mean(dim=1)  # average across time frames
classifier = nn.Linear(256, 3)
logits = classifier(pooled)
probabilities = torch.softmax(logits, dim=-1)

print('Pooled sequence:', pooled.shape)
print('Logits:', logits.shape, logits.detach())
print('Softmax probabilities (UNTRAINED):', probabilities.detach())
print('Probability sum:', probabilities.sum(dim=-1).detach())

## 10. Recap: all the shapes in one place

These are the actual shapes for the **two-second concatenated** recording in this notebook. If you switch to the one-second *mixed* recording, time-frame counts change from **201** to **101**.

| Stage | Shape | Meaning |
|---|---|---|
| Waveform | `[32000]` | 32,000 amplitude samples = 2 seconds at 16 kHz |
| STFT magnitude | `[201, 201]` | 201 FFT frequency bins × 201 time frames |
| Mel spectrogram | `[80, 201]` | 80 Mel bands × 201 time frames |
| Transposed Mel frames | `[201, 80]` | 201 frames, each with 80 continuous features |
| Linear projection | `[201, 256]` | 256-dimensional embedding for each frame |
| + position vectors | `[201, 256]` | Same dimensionality, now with frame-order information |
| + batch dimension | `[1, 201, 256]` | 1 recording in the batch |
| Self-attention output | `[1, 201, 256]` | Contextualized frame embeddings |
| Mean pooling | `[1, 256]` | One summary vector for the recording |
| Classifier | `[1, 3]` | Three untrained class logits |

**Big picture:** Unlike text tokenization, our path uses **continuous acoustic feature vectors**. Some other audio models instead use raw waveforms or discrete codec tokens. This notebook shows one simple way to get audio into a Transformer.

**What this notebook does not do:** train model weights, recognize spoken words, or generate speech. Those require datasets, objectives/losses, and appropriate output architectures (e.g., ASR decoders).

## 11. Self-check (no code required)

1. If we keep a 16 kHz sample rate but record 3 seconds, how many samples are in the waveform?
2. What's the difference between **adding** 440 Hz and 880 Hz waves and **concatenating** them?
3. What does `hop_length=160` control?
4. If `n_mels=40`, what is the Mel tensor shape for our two-second waveform?
5. Why do we transpose `[80, 201]` into `[201, 80]`?
6. Does the final classifier understand the difference between speech, music, and noise already? Why or why not?

**Suggested next lesson:** Use a *real audio recording*, then learn how a trainable loss updates the projection, attention, and output layers.